# CHALLENGE 1: CONNECTING TO DATABASE

In [94]:
import redshift_connector

# Create a connection to the Redshift database with the above credentials
conn = redshift_connector.connect(
    host='c23-workgroup.129033205317.eu-west-2.redshift-serverless.amazonaws.com',
    database='dw_air_travel',
    user='yusuf_ahmed',
    password='Dyusuf_71',
    port=5439
)
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')

# Extract the result from the cursor object
output = cursor.fetch_dataframe()

print(output)

# Close the cursor
cursor.close()

   count
0    112


### How many Distinct laser colours are there? 84

In [7]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct laser_colour) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     84


### How many distinct simplified laser colours? 71

In [3]:
# Create a cursor to use to interact with the database
cursor = conn.cursor()
# Execute an SQL query
cursor.execute('SELECT Count(Distinct (Lower(laser_colour))) FROM laser_incident')
# Extract the result from the cursor object
output = cursor.fetch_dataframe()
print(output)
# Close the cursor
cursor.close()

   count
0     71


## TASK 2: Extracting Data Into Pandas

In [4]:
import pandas as pd
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM laser_incident
            JOIN airport ON laser_incident.airport_id = airport.airport_id
            JOIN city ON airport.city_id = city.city_id
            JOIN state ON city.state_id = state.state_id
            WHERE state_name = 'Michigan'
        """)

        michigan_laser = cursor.fetch_dataframe()
        print("Number of rows:", len(michigan_laser))

    except Exception as e:
        conn.rollback()
        print(e)

Number of rows: 418


## Task 3
### How many rows are in the dataframe? 418
### What percentage of laser incidents cause significant harm? 0.72%
### What airport has the worst problem with laser incidents? DETROIT METRO WAYNE COUNTY = 176
### How has the frequency of laser incidents changed over time? Decreased slightly over time
### What are the most and least common laser colours used? Green as the Most, Orange as the least

In [5]:
injury = michigan_laser["injury"].value_counts()
print(injury)

injury
f    415
t      3
Name: count, dtype: int64


In [6]:
airport_counts = michigan_laser['airport_name'].value_counts()
print(airport_counts)
michigan_laser.info

airport_name
DETROIT METRO WAYNE COUNTY          176
KALAMAZOO/BATTLE CREEK INTL          37
GERALD R FORD INTL                   35
CAPITAL REGION INTL                  24
BISHOP INTL                          19
MBS INTL                             17
COLEMAN A YOUNG MUNI                 17
OAKLAND COUNTY INTL                  15
CHERRY CAPITAL                       12
MUSKEGON COUNTY                      10
JACKSON COUNTY/REYNOLDS FLD           5
BATTLE CREEK EXEC AT KELLOGG FLD      5
DELTA COUNTY                          4
WILLOW RUN                            4
SELFRIDGE ANGB                        3
PADGHAM FLD                           2
ST CLAIR COUNTY INTL                  2
ALPENA COUNTY RGNL                    2
HILLSDALE MUNI                        2
KIRSCH MUNI                           2
MOUNT PLEASANT MUNI                   2
OTTAWA EXEC                           2
BRANCH COUNTY MEML                    1
CLARE COUNTY                          1
DUPONT/LAPEER              

<bound method DataFrame.info of      laser_incident_id flight_id aircraft  altitude laser_colour injury  \
0                28554    DAL859     A319      8000        Green      f   
1                29885    N350KL     CL35      3000         Blue      f   
2                11781   RPA3527     E170     20000        Green      f   
3                26004     N11SP    HELO       1700        Green      f   
4                17595    DAL853     A321      2000        Green      f   
..                 ...       ...      ...       ...          ...    ...   
413              10699    N383ST     B407      2500         Blue      f   
414              15254   ENY3599     E145      4000        Green      f   
415              27179   SWA1486     B737      4000        Green      f   
416              19509    NKS852     A320      4500        Green      f   
417               3021    N421SC     F2TH      2300         Blue      f   

     airport_id                  at  airport_id                 air

In [7]:
import altair as alt
alt.data_transformers.enable("vegafusion")

michigan_laser = michigan_laser.loc[:, ~michigan_laser.columns.duplicated()]

chart_data = (
    michigan_laser
    .assign(year=michigan_laser['at'].dt.to_period('Y').astype(str))
)

alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('year', title='Year', sort='x'),
    y=alt.Y('count()', title='Total Incidents'),
    tooltip=[
        alt.Tooltip('year', title='Year'),
        alt.Tooltip('count()', title='Total Incidents')
    ]
).properties(
    width=800,
    height=400,
    title='Incidents over time'
)

alt.Chart(...)

In [8]:
alt.Chart(michigan_laser).mark_bar().encode(
    x=alt.X("laser_colour", title="Colour of Laser", sort="-y"),
    y=alt.Y("count()", title="Count"),
    color=alt.Color('laser_colour')
).properties(
    title="Different laser colours",
    width=600,
    height=400)

alt.Chart(...)

# CHALLENGE 2: VISUALISATION

## TASK 1: PIE CHARTS

In [ ]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            WITH most_active_airline AS (
                SELECT op_unique_carrier
                FROM flight
                WHERE op_unique_carrier IN (SELECT iata FROM airline)
                GROUP BY op_unique_carrier
                ORDER BY COUNT(*) DESC
                LIMIT 1
            )
            SELECT
                f.tail_num,
                CASE 
                    WHEN COALESCE(CAST(NULLIF(TRIM(f.arr_delay), '') AS FLOAT), 0) > 0
                        THEN 'Delayed'
                    ELSE 'On Time'
                END AS delayed
            FROM flight f
            JOIN most_active_airline m
                ON f.op_unique_carrier = m.op_unique_carrier;
        """)

        flight_delayed = cursor.fetch_dataframe()
        print(flight_delayed)

    except Exception as e:
        conn.rollback()
        print(e)


    

       tail_num  delayed
0        N8526W  On Time
1        N234WN  On Time
2        N420WN  On Time
3        N436WN  On Time
4        N788SA  Delayed
...         ...      ...
127768   N8757L  On Time
127769   N8717M  On Time
127770   N7733B  On Time
127771   N7828A  Delayed
127772   N8822Q  Delayed

[127773 rows x 2 columns]


In [11]:
import altair as alt
delay_counts = flight_delayed['delayed'].value_counts()
delay_percent = (delay_counts / delay_counts.sum()) * 100
delay_percent = delay_percent.reset_index()
delay_percent.columns = ['status', 'percentage']


alt.Chart(delay_percent).mark_arc().encode(
    theta=alt.Theta(field="percentage", type="quantitative"),
    color=alt.Color(field="status", type="nominal"),
    tooltip=[
        alt.Tooltip("status", title="Status"),
        alt.Tooltip("percentage", format=".2f")
    ]
).properties(
    title="Percentage of Flights Delayed (Most Active Airline)"
)

alt.Chart(...)

## Task 2: Emphasis

In [12]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT
                SUM(CAST(NULLIF(TRIM(carrier_delay), '') AS FLOAT)) AS total_carrier_delay,
                SUM(CAST(NULLIF(TRIM(weather_delay), '') AS FLOAT)) AS total_weather_delay,
                SUM(CAST(NULLIF(TRIM(nas_delay), '') AS FLOAT)) AS total_nas_delay,
                SUM(CAST(NULLIF(TRIM(security_delay), '') AS FLOAT)) AS total_security_delay,
                SUM(CAST(NULLIF(TRIM(late_aircraft_delay), '') AS FLOAT)) AS total_late_aircraft_delay
            FROM flight
            WHERE TRIM(cancelled) = '0.0';
        """)
        delay_totals = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)

delay_chart_data = pd.DataFrame({
    "delay_type": [
        "Carrier Delay",
        "Weather Delay",
        "NAS Delay",
        "Security Delay",
        "Late Aircraft Delay"
    ],
    "minutes_lost": [
        delay_totals.loc[0, "total_carrier_delay"],
        delay_totals.loc[0, "total_weather_delay"],
        delay_totals.loc[0, "total_nas_delay"],
        delay_totals.loc[0, "total_security_delay"],
        delay_totals.loc[0, "total_late_aircraft_delay"]
    ]
})

# highlight the biggest cause
max_delay = delay_chart_data["minutes_lost"].max()
delay_chart_data["highlight"] = delay_chart_data["minutes_lost"] == max_delay

alt.Chart(delay_chart_data).mark_bar().encode(
    y=alt.Y(
        "delay_type:N",
        sort="-x",
        title=None
    ),
    x=alt.X(
        "minutes_lost:Q",
        title="Total Minutes Lost"
    ),
    color=alt.condition(
        alt.datum.highlight,
        alt.value("#d62728"),
        alt.value("#9ecae1")
    ),
    tooltip=[
        alt.Tooltip("delay_type:N", title="Delay Type"),
        alt.Tooltip("minutes_lost:Q", title="Minutes Lost", format=",")
    ]
).properties(
    width=700,
    height=300,
    title="Total Minutes Lost by Delay Type"
)

alt.Chart(...)

## Task 3: Justification

### What percentage of laser incidents cause significant harm? Used a pie chart to show the percentage in comparison of harmful and non harmful incidents, makes it easier to understand the proportion as its a circle.

In [20]:
injury_status = michigan_laser["injury"].value_counts()
print(injury_status)

injury_percent = (injury_status / injury_status.sum()) * 100
harm_df = injury_percent.reset_index()
harm_df.columns = ['injury', 'percentage']

alt.Chart(harm_df).mark_arc().encode(
    theta=alt.Theta(field="percentage", type="quantitative"),
    color=alt.Color(field="injury", type="nominal"),
    tooltip=[
        alt.Tooltip("injury", title="Harmful?"),
        alt.Tooltip("percentage", format=".2f")
    ]
).properties(
    title="Percentage of harmful injuries from laser"
)

injury
f    415
t      3
Name: count, dtype: int64


alt.Chart(...)

### What airport has the worst problem with laser incidents? Similar to the laser colour graph, I used a bar chart as it is discrete categorical x axis values, so this would make it easy to see the worst airport

In [17]:
airport_counts = michigan_laser['airport_name'].value_counts()

alt.Chart(michigan_laser).mark_bar().encode(
    x=alt.X("airport_name", title="airport", sort="-y"),
    y=alt.Y("count()", title="Number of incidents"),
    color=alt.Color('airport_name'),
    tooltip=[
        alt.Tooltip('airport_name', title='Airport'),
        alt.Tooltip('count()', title='Number of Incidents')
    ]
).properties(
    title="Different laser colours",
    width=600,
    height=400)

alt.Chart(...)

### How has the frequency of laser incidents changed over time? Used a line graph to really show the change of the frequency of incidents over time.

In [13]:
alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('year', title='Year', sort='x'),
    y=alt.Y('count()', title='Total Incidents'),
    tooltip=[
        alt.Tooltip('year', title='Year'),
        alt.Tooltip('count()', title='Total Incidents')
    ]
).properties(
    width=800,
    height=400,
    title='Incidents over time'
)

alt.Chart(...)

### What are the most and least common laser colours used? I used a bar chart as we have discrete values which are non numerical on the x axis, so the bar chart is the most suitable to produce a clear representation

In [14]:
alt.Chart(michigan_laser).mark_bar().encode(
    x=alt.X("laser_colour", title="Colour of Laser", sort="-y"),
    y=alt.Y("count()", title="Count"),
    color=alt.Color('laser_colour')
).properties(
    title="Different laser colours",
    width=600,
    height=400)

alt.Chart(...)

## PHASE 3: Data Import

### Task 1: Extract the data

In [61]:
new_laser_report = pd.read_excel(
    "/Users/yusufahmed/Documents/PreCourseLearning/week8_data_analysis_2/Workshop/Coursework_Data_Analysis_Fundamentals_2/Laser-Report-2021-FOIA-FINAL.xlsx")
new_laser_report.info()
print(new_laser_report.head(5))

<class 'pandas.DataFrame'>
RangeIndex: 9723 entries, 0 to 9722
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Incident Date  9723 non-null   datetime64[us]
 1   Incident Time  9723 non-null   int64         
 2   Aircraft       9376 non-null   str           
 3   Altitude       9723 non-null   object        
 4   Airport        9723 non-null   str           
 5   Laser Color    9723 non-null   str           
 6   Injury         9723 non-null   str           
 7   City           9722 non-null   str           
 8   State          9722 non-null   str           
dtypes: datetime64[us](1), int64(1), object(1), str(6)
memory usage: 979.3+ KB
  Incident Date  Incident Time Aircraft Altitude Airport Laser Color Injury  \
0    2021-01-01             48     E75L     2000     SAV       Green     No   
1    2021-01-01             56     E75L    17000     ZOA       Green     No   
2    2021-01-01            1

### Task 2: Data Exploration
The main issues I notice are:
- The date and time column are seperated and not in one column together
- Altitude is not a number datatype
- Injury column has yes and no and not 't' and 'f'
- No flight_id

### Task 3: Transform the Data

In [62]:
# Padded the incident time so it can be put changed to the at column
new_laser_report['Incident Time'] = (
    new_laser_report['Incident Time']
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

new_laser_report['at'] = pd.to_datetime(
    new_laser_report['Incident Date'].astype(str) + ' ' + new_laser_report['Incident Time'].astype(str),
    errors='coerce'
)

new_laser_report = new_laser_report.drop(columns=['Incident Date', 'Incident Time'])
new_laser_report.head()

,Aircraft,Altitude,Airport,Laser Color,Injury,City,State,at
0,E75L,2000,SAV,Green,No,Savannah,Georgia,2021-01-01 00:48:00
1,E75L,17000,ZOA,Green,No,Fremont,California,2021-01-01 00:56:00
2,C208,5000,DSM,Green,No,Des Moines,Iowa,2021-01-01 01:02:00
3,E120,2000,ONT,Green,No,Ontario,California,2021-01-01 01:25:00
4,BE9L,4000,ZSE,Green,No,Auburn,Washington,2021-01-01 01:40:00


In [63]:
# Cleaning column names
new_laser_report = new_laser_report.rename(columns={
    'Aircraft': 'aircraft',
    'Altitude': 'altitude',
    'Airport': 'airport_code',
    'Laser Color': 'laser_colour',
    'Injury': 'injury',
    'City': 'city_name',
    'State': 'state_name'
})

#Made altitude a number rather than an object
new_laser_report['altitude'] = pd.to_numeric(
    new_laser_report['altitude'],
    errors='coerce'
)


In [65]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM s_yusuf_ahmed.airport;
        """)
        airport_df = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)


# Joins the airport table
new_laser_report = new_laser_report.merge(
    airport_df[['airport_id', 'airport_code']],
    on='airport_code',
    how='left'
)
# Drop column we do not need
new_laser_report = new_laser_report.drop(columns=[
    'airport_id_x',
    'airport_id_y',
    'airport_code',
    'city_name',
    'state_name'
], errors='ignore')

# Removes airport Id not in the airport warehouse
new_laser_report = new_laser_report[
    new_laser_report['airport_id'].notna()
].copy()


new_laser_report.info()

KeyError: 'airport_code'

In [67]:
new_laser_report.info()

<class 'pandas.DataFrame'>
Index: 4587 entries, 0 to 9722
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   aircraft      4381 non-null   str           
 1   altitude      4514 non-null   float64       
 2   laser_colour  4587 non-null   str           
 3   injury        4587 non-null   str           
 4   at            4587 non-null   datetime64[us]
 5   airport_id    4587 non-null   float64       
dtypes: datetime64[us](1), float64(2), str(3)
memory usage: 301.1 KB


In [ ]:
#Cleaning up unstructured values and data
new_laser_report['airport_id'] = new_laser_report['airport_id'].astype(int)
new_laser_report['aircraft'] = new_laser_report['aircraft'].fillna('unknown')
new_laser_report['altitude'] = new_laser_report['altitude'].fillna(0)
new_laser_report['injury'] = (
    new_laser_report['injury']
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        'yes': 't',
        'no': 'f',
        'unk': 'f'
    })
)
new_laser_report['injury'].value_counts()

injury
f    4561
t      26
Name: count, dtype: int64

### Task 4: Load the Data

In [ ]:
# Drops the flight_id from the laser_incident as it does not link to anything
query = """ALTER TABLE s_yusuf_ahmed.laser_incident
    DROP COLUMN flight_id;"""

with conn.cursor() as cursor:
    try:
        cursor.execute(query)
        laser_incident = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)



/var/folders/s0/m30t_y5j5nq20xptjt3ttndr0000gn/T/ipykernel_60897/2653459960.py:7: UserWarning: No row description was found. pandas dataframe will be missing column labels.
  laser_incident = cursor.fetch_dataframe()


no result set


In [ ]:
# FInds the max id in the warehouse table to then generate new ids
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT COALESCE(MAX(laser_incident_id), 0) AS max_id
        FROM s_yusuf_ahmed.laser_incident
    """)
    max_id = cursor.fetch_dataframe().loc[0, 'max_id']
    print(max_id)

new_laser_report = new_laser_report.reset_index(drop=True)

new_laser_report['laser_incident_id'] = range(
    int(max_id) + 1,
    int(max_id) + 1 + len(new_laser_report)
)


31502


In [90]:
new_laser_report.tail()

,aircraft,altitude,laser_colour,injury,at,airport_id,laser_incident_id
4582,BE99,2000.0,Green,f,2021-12-31 02:04:00,14124,36085
4583,B747,900.0,Green,f,2021-12-31 02:29:00,15005,36086
4584,B407,1500.0,Green,f,2021-12-31 03:56:00,16249,36087
4585,B763,3200.0,Green,f,2021-12-31 18:59:00,2406,36088
4586,B407,800.0,Green,f,2021-12-31 22:00:00,11635,36089


In [89]:
# Inserting the data into the table
with conn.cursor() as cursor:
    try:
        insert_query = """
            INSERT INTO s_yusuf_ahmed.laser_incident
            (laser_incident_id,aircraft, altitude, laser_colour, injury, airport_id, at)
            VALUES (%s,%s, %s, %s, %s, %s, %s)
        """

        for _, row in new_laser_report.iterrows():
            cursor.execute(insert_query, (
                int(row['laser_incident_id']),
                row['aircraft'],
                row['altitude'],
                row['laser_colour'],
                row['injury'],
                int(row['airport_id']),
                row['at']
            ))

        conn.commit()
        print("Data inserted successfully!")

    except Exception as e:
        conn.rollback()
        print("Error:", e)

Data inserted successfully!


## PHASE 5: ANOMALY DETECTION

### Which airline has the most delays? Is this an outlier? Is there an explanation? 
Southwest Airlines (WN) has the highest total number of delayed flights (45,848). However, this is largely driven by the fact that it operates significantly more flights than any other airline. So, this is not an outlier

In [96]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM flight
            WHERE TRIM(cancelled) = '0.0';
        """)
        flight_df = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)

Exception ignored while finalizing file <_io.BufferedRWPair object at 0x140c96dc0>:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/socket.py", line 743, in write
    return self._sock.send(b)
  File "/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/ssl.py", line 1232, in send
    return self._sslobj.write(data)
ssl.SSLError: Underlying socket connection gone (_ssl.c:2480)


In [99]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM airline;
        """)
        airline_df = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)

In [101]:
flight_df['arr_delay'] = pd.to_numeric(flight_df['arr_delay'], errors='coerce')

# Keep only valid airlines
valid_airlines = airline_df['iata']

filtered = flight_df[flight_df['op_unique_carrier'].isin(valid_airlines)]

# Create delay flag
filtered['is_delayed'] = filtered['arr_delay'] > 0

# Count delays per airline
delays_by_airline = (
    filtered.groupby('op_unique_carrier')
    .agg(
        total_flights=('op_unique_carrier', 'count'),
        delayed_flights=('is_delayed', 'sum')
    )
    .reset_index()
)

delays_by_airline['delay_rate'] = (
    delays_by_airline['delayed_flights'] / delays_by_airline['total_flights']
)

delays_by_airline.sort_values('delayed_flights', ascending=False)

,op_unique_carrier,total_flights,delayed_flights,delay_rate
9,WN,126967,45848,0.361102
0,AA,75372,26171,0.347224
3,DL,81000,16891,0.208531
8,UA,59767,16524,0.276474
2,B6,21315,9104,0.427117
7,NK,22485,9000,0.400267
1,AS,19272,6768,0.351183
4,F9,15622,5769,0.369287
6,HA,6602,3120,0.472584
5,G4,9418,3025,0.321193


### Task 2: Search through the laser_incident and flight tables. Can you identify any anomalies?
1. Delays greater than or equal to 1,440 minutes (24 hours) were excluded from the analysis, as these are unlikely to represent genuine flight delays and are more likely due to data errors or misclassified cancellations. This threshold ensures that the analysis focuses on realistic operational delays.
2. A value of 240,000 ft is still unrealistic. Commercial aircraft typically operate below 40,000 ft, and even high-performance military aircraft rarely exceed 80,000 ft. Therefore, altitude values above 60,000 ft were considered invalid and removed from the dataset.

In [106]:
with conn.cursor() as cursor:
    try:
        cursor.execute("""
            SELECT *
            FROM laser_incident;
        """)
        laser_incident = cursor.fetch_dataframe()
    except Exception as e:
        conn.rollback()
        print(e)

In [107]:
laser_incident.describe()


,laser_incident_id,altitude,airport_id,at
count,36303.000000,3.630300e+04,36303.000000,36303
mean,18124.904223,7.180016e+03,8721.652425,2018-11-11 20:10:37.087844
min,1.000000,0.000000e+00,68.000000,2016-01-01 00:07:00
25%,9076.500000,2.500000e+03,2578.000000,2017-04-27 03:40:00
50%,18152.000000,5.000000e+03,7208.000000,2018-10-31 07:22:00
75%,27227.500000,9.000000e+03,14738.000000,2020-05-21 02:19:30
max,36089.000000,1.350020e+06,19957.000000,2021-12-31 22:00:00
std,10439.090232,1.047238e+04,5986.514043,NaN


In [105]:
flight_df.describe()

,index_nr,arr_delay,rn
count,568084.000000,566941.000000,568084.0
mean,284934.154805,0.509332,1.0
std,164725.680356,49.070609,0.0
min,0.000000,-98.000000,1.0
25%,142279.750000,-17.000000,1.0
50%,284545.500000,-8.000000,1.0
75%,427471.250000,5.000000,1.0
max,570405.000000,3795.000000,1.0


In [115]:
clean_flights = flight_df[flight_df['arr_delay'].notna()]
top_10_delays = (
    clean_flights
    .sort_values('arr_delay', ascending=False)
    .head(20)
)

top_10_delays[['arr_delay']]

,arr_delay
227655,3795.0
4516,2924.0
367730,2604.0
543300,2484.0
108763,2427.0
165687,2403.0
253459,2289.0
94262,2028.0
514971,1873.0
216446,1812.0


In [116]:
#Filtering out flights greater than 24 hours delayed
filtered_flights = flight_df[
    (flight_df['arr_delay'].notna()) &
    (flight_df['arr_delay'] < 1440)
]
clean_flights = filtered_flights[filtered_flights['arr_delay'].notna()]
top_10_delays = (
    clean_flights
    .sort_values('arr_delay', ascending=False)
    .head(20)
)

top_10_delays[['arr_delay']]

,arr_delay
44585,1428.0
15800,1423.0
168646,1410.0
323177,1396.0
195023,1364.0
177078,1359.0
281579,1356.0
475946,1355.0
436900,1352.0
138164,1338.0


In [ ]:
# Filtered the altitude so it is all less than 60000 ft as that is a realistic height for the planes to go to.
filtered_laser_incident = laser_incident[
    (laser_incident['altitude'] > 0) &
    (laser_incident['altitude'] < 60000)
]
top_20_altitudes = (
    filtered_laser_incident
    .sort_values('altitude', ascending=False)
    .head(20)
)

print(top_20_altitudes)

       laser_incident_id aircraft  altitude laser_colour injury  airport_id  \
12604               1392     E75L     55000        Green      f       16373   
25308               5540     CRUZ     55000        Green      f       17098   
25687                545     B77W     50000        Green      f        2182   
14720               1429     B738     50000         Blue      f        2410   
17965               1723     HH60     50000        Green      f        4672   
29617                383     B738     50000        Green      f        2413   
23694                544     B739     50000        Green      f        2182   
14953              29756    HELO      50000        Green      f        3662   
19143              25980     C172     50000         UNKN      f       15005   
35666              26337     C172     50000        Green      f        3529   
15667                734     P28A     50000        Green      f        2625   
19816              12030     CRJ2     50000        G